# Intialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType, DoubleType, IntegerType
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import StringType

# Read From Bronze

In [0]:
df = spark.table("workspace.bronze.sales_details")

# Exploring Data

In [0]:
print("4️⃣ عينة من البيانات (اضغط على Data Profile):")
display(df.limit(100))

# 1. فحص الهيكل وأنواع البيانات
print("1️⃣ هيكل الجدول وأنواع البيانات (Schema & Data Types):")
df.printSchema()
print("-" * 50)

# 2. فحص التكرار (Duplicates Check)
total_rows = df.count()
distinct_rows = df.distinct().count()
print("2️⃣ فحص التكرار:")
print(f"- إجمالي الصفوف: {total_rows}")
print(f"- الصفوف المكررة: {total_rows - distinct_rows} صف مكرر")
print("-" * 50)

# 3. تقرير القيم المفقودة (Nulls)
print("3️⃣ تقرير القيم المفقودة (Nulls):")
display(df.pandas_api().isnull().sum(axis=0))
print("-" * 50)



# Transformations

## Cleaning Dates

In [0]:
# بنلقط كل عواميد التواريخ أوتوماتيك
date_cols = [c for c in df.columns if 'dt' in c]

# بنطبق لوجيك السنيور (الشرط الصريح) بس على كل العواميد بـ Loop مختصرة
for c in date_cols:
    df = df.withColumn(
        c,
        F.when(
            (F.col(c) == 0) | (F.length(F.col(c).cast("string")) != 8), 
            F.lit(None)
        ).otherwise(F.expr(f"try_to_date(cast({c} as string), 'yyyyMMdd')"))
    )

## Trimming

In [0]:
trim_exprs = [
    F.trim(F.col(c)).alias(c) if t == 'string' else F.col(c)
    for c, t in df.dtypes
]
df = df.select(*trim_exprs)

## Rename Columns


In [0]:
Rename_Map={
'sls_ord_num':'order_number', 
'sls_prd_key':'product_key',     
'sls_cust_id':'customer_id',     
'sls_order_dt':'order_date',    
'sls_ship_dt':'ship_date',     
'sls_due_dt':'due_date',      
'sls_sales':'sales_amount',       
'sls_quantity':'sales_quantity',    
'sls_price':'sales_price'       
}

for Old_Name,New_Name in Rename_Map.items():
    df=df.withColumnRenamed(Old_Name,New_Name)

## Sales & Price Correction


In [0]:
df = (
    df
    # 1. تحويل الأنواع
    .withColumn("sales_quantity", F.col("sales_quantity").cast(IntegerType()))
    .withColumn("sales_price", F.col("sales_price").cast(DoubleType()))
    .withColumn("sales_amount", F.col("sales_amount").cast(DoubleType()))
    
    # 2. ضمان صحة الحسابات ومعالجة القيم الفارغة/السالبة
    .withColumn("sales_price", F.abs(F.coalesce(F.col("sales_price"), F.lit(0.0))))
    .withColumn("sales_quantity", F.abs(F.coalesce(F.col("sales_quantity"), F.lit(0))))
    .withColumn(
        "sales_amount", 
        F.when(
            (F.col("sales_amount").isNull()) | (F.col("sales_amount") <= 0),
            F.col("sales_quantity") * F.col("sales_price")
        ).otherwise(F.abs(F.col("sales_amount")))
    )
)

## Sanity Checks Of DataFrame

In [0]:
df.limit(100).display()

# Write Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("workspace.silver.crm_sales")
)

## Sanity Check Of Silver Table


In [0]:
%sql
select * from workspace.silver.crm_sales limit 10;